# DQN on unity environment

Exploring the unity environment and Solving **Basic** example.

In [1]:
import mlagents_envs
from mlagents_envs.environment import UnityEnvironment
import numpy as np

In [ ]:
env = UnityEnvironment(file_name=None)  
#open the scenes on editor and press play

In [3]:
env.reset()

## Behavior

A behavior is a named policy, not an agent. In Unity, every Agent component has Behavior Parameters with a Behavior Name; all agents sharing that name share one policy and get batched together on the Python side. So one environment can expose several behaviors (say "Striker" and "Goalie"), and each behavior can be driven by many agents at once.

In [4]:
print(list(env.behavior_specs))

['Basic?team=0']


## BehaviorSpec

This is the ML-Agents equivalent of observation_space and action_space:

In [5]:
# We will only consider the first Behavior
behavior_name = list(env.behavior_specs)[0]
print(f"Name of the behavior : {behavior_name}")
spec = env.behavior_specs[behavior_name]

Name of the behavior : Basic?team=0


Observations are a list, not a single array. An agent can have several sensors — a vector sensor, a camera, a raycast sensor — and each is a separate entry with its own shape. 
Vector observations come back as (n_agents, dim); visual ones as (n_agents, height, width, channels), which is NHWC, so you'll need a permute before feeding a PyTorch conv layer.


Actions are hybrid. ActionSpec carries both a continuous_size and discrete_branches simultaneously. discrete_branches is a tuple giving the number of choices per branch — so (3,)
is a single 3-way choice, while (3, 2) means two independent decisions made each step. For DQN you want a single branch; multi-branch needs either a factored Q-head or flattening the joint space.

In [6]:
for obs_spec in spec.observation_specs:
    print(obs_spec.name, obs_spec.shape, obs_spec.observation_type)

print("continuous:", spec.action_spec.continuous_size)
print("discrete branches:", spec.action_spec.discrete_branches)

Basic (20,) ObservationType.DEFAULT
continuous: 0
discrete branches: (3,)


In [7]:
# Examine the number of observations per Agent
print("Number of observations : ", len(spec.observation_specs))

# Is there a visual observation ?
# Visual observation have 3 dimensions: Height, Width and number of channels
vis_obs = any(len(sp.shape) == 3 for sp in spec.observation_specs)
print("Is there a visual observation ?", vis_obs)

Number of observations :  1
Is there a visual observation ? False


In [8]:
# Is the Action continuous or multi-discrete ?
if spec.action_spec.continuous_size > 0:
  print(f"There are {spec.action_spec.continuous_size} continuous actions")
if spec.action_spec.is_discrete():
  print(f"There are {spec.action_spec.discrete_size} discrete actions")


# How many actions are possible ?
#print(f"There are {spec.action_size} action(s)")

# For discrete actions only : How many different options does each action has ?
if spec.action_spec.discrete_size > 0:
  for action, branch_size in enumerate(spec.action_spec.discrete_branches):
    print(f"Action number {action} has {branch_size} different options")



There are 1 discrete actions
Action number 0 has 3 different options


This is the real departure from Gym. env.step() does not take an action and does not return an observation. It advances the simulation until at least one agent requests a decision. You then query what happened:

DecisionSteps holds agents that need an action right now. Fields: obs (list of arrays), reward, agent_id, action_mask, group_id, group_reward.

TerminalSteps holds agents whose episode ended on this step. Same fields, minus action_mask, plus interrupted.

An agent can appear in both in the same step — it ended an episode and its replacement already needs an action.

The reward is accumulated since that agent's last decision, not since the last env.step(). With a Decision Requester period greater than 1, several physics frames fold into one transition. That's correct behavior, but it means your effective discount factor is per-decision, not per-frame.

There's also no auto-reset returning an initial observation. Unity resets agents internally; the new episode's first observation just shows up in a later DecisionSteps.

In [9]:
decision_steps, terminal_steps = env.get_steps(behavior_name)

## The indexing trap

Row order in the obs array is not agent_id, and it changes between steps as agents terminate and respawn. Mixing them up scrambles your transitions in a way that still trains, just badly — which makes it painful to debug.

In [10]:
agent_id = list(decision_steps.agent_id)[0]
idx = decision_steps.agent_id_to_index[agent_id]
obs_for_that_agent = decision_steps.obs[0][idx]

print(f"Agent id : {agent_id} idx : {idx} observation : {obs_for_that_agent}")

Agent id : 0 idx : 0 observation : [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


Or use the single-agent accessor, which handles it for you:

In [11]:
step = decision_steps[agent_id]   # returns one DecisionStep
step.obs, step.reward, step.action_mask

([array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.,
         0., 0., 0.], dtype=float32)],
 0.0,
 [array([False, False, False])])

## Sending actions

In [12]:
from mlagents_envs.environment import ActionTuple
import numpy as np

n = len(decision_steps)
actions = np.random.randint(0, 3, size=(n, 1), dtype=np.int32)
env.set_actions(behavior_name, ActionTuple(discrete=actions))
env.step()

Shapes are strict: discrete must be (n_agents, n_branches) as int32, continuous must be (n_agents, continuous_size) as float32. Row i corresponds to row i of the observations you just read from DecisionSteps — so build your action array in that same order.

Two helpers save time:

In [13]:
spec.action_spec.empty_action(n)    # all zeros, correct shape and dtype
spec.action_spec.random_action(n)   # valid random actions

Action masking

For discrete actions, decision_steps.action_mask is a list with one boolean array per branch, shaped (n_agents, branch_size). True means the action is unavailable — the polarity catches people out. Apply it before your argmax:

```
q = q_net(obs_tensor)            # (n, n_actions)
mask = torch.as_tensor(decision_steps.action_mask[0])
q = q.masked_fill(mask, -float("inf"))
greedy = q.argmax(dim=1)
```

In [14]:
all_rewards = []
for episode in range(30):
  env.reset()
  decision_steps, terminal_steps = env.get_steps(behavior_name)
  tracked_agent = -1 # -1 indicates not yet tracking
  done = False # For the tracked_agent
  episode_rewards = 0 # For the tracked_agent
  while not done:
    # Track the first agent we see if not tracking
    # Note : len(decision_steps) = [number of agents that requested a decision]
    if tracked_agent == -1 and len(decision_steps) >= 1:
      tracked_agent = decision_steps.agent_id[0]

    # Generate an action for all agents
    action = spec.action_spec.random_action(len(decision_steps))

    # Set the actions
    env.set_actions(behavior_name, action)

    # Move the simulation forward
    env.step()

    # Get the new simulation results
    decision_steps, terminal_steps = env.get_steps(behavior_name)
    if tracked_agent in decision_steps: # The agent requested a decision
      episode_rewards += decision_steps[tracked_agent].reward
    if tracked_agent in terminal_steps: # The agent terminated its episode
      episode_rewards += terminal_steps[tracked_agent].reward
      done = True
      all_rewards.append(episode_rewards)
      print(f"\rTotal rewards for episode {episode} is {episode_rewards:.4f}", end="")


Total rewards for episode 29 is -0.5900

In [15]:
print(f"\nAverage rewards over {len(all_rewards)} episodes is {np.mean(all_rewards):.4f}")


Average rewards over 30 episodes is 0.0517


In [16]:
env.close()
print("Closed environment")


Closed environment


UnityToGymWrapper in mlagents_envs.envs converts a single-agent behavior into a standard Gym env, which lets you drop Basic straight into an existing DQN implementation.
The cost is that it hides the multi-agent batching, so you lose the free parallelism when you move to environments with many agents in one scene.

In [ ]:
##new env

env = UnityEnvironment(file_name=None)  #play on editor


In [ ]:
from mlagents_envs.envs.unity_gym_env import UnityToGymWrapper

env = UnityToGymWrapper(env, uint8_visual=True)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[WARNING] uint8_visual was set to true, but visual observations are not in use. This setting will not have any effect.
[WARNING] The environment contains multiple observations. You must define allow_multiple_obs=True to receive them all. Otherwise, only the first visual observation (or vector observation ifthere are no visual observations) will be provided in the observation.


In [46]:
env.observation_space, env.action_space, env.action_size

(Box(-inf, inf, (20,), float32), Discrete(3), 1)

In [49]:
for episode in range(30):
  env.reset()

  done = False
  episode_rewards = 0
  while not done:
    action = env.action_space.sample()
    obs, reward, done, info = env.step(action)
    episode_rewards += reward
    if done:
        print(f"\rTotal rewards for episode {episode} is {episode_rewards:.4f}", end="")

Total rewards for episode 29 is 0.05000